In [ ]:
import requests
from bs4 import BeautifulSoup

def fetch_pubmed_articles_with_metadata(query: str, max_results=3, use_mock_if_empty=True):
    headers = {"User-Agent": "Mozilla/5.0"}

    # Step 1: Search PubMed
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    search_params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json"
    }
    try:
        search_response = requests.get(search_url, params=search_params, headers=headers, timeout=10).json()
        id_list = search_response["esearchresult"]["idlist"]
        print("Found PubMed IDs:", id_list)
        if not id_list:
            raise ValueError("No IDs found for this query.")

        ids = ",".join(id_list)

        # Step 2: Fetch article summaries
        fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        fetch_params = {
            "db": "pubmed",
            "id": ids,
            "retmode": "xml"
        }
        fetch_response = requests.get(fetch_url, params=fetch_params, headers=headers, timeout=10)
        try:
            soup = BeautifulSoup(fetch_response.text, "lxml")
        except:
            soup = BeautifulSoup(fetch_response.text, "html.parser")
        articles_xml = soup.find_all("pubmedarticle")
        print("Articles found in XML:", len(articles_xml))

        articles_info = []
        for article, pmid in zip(articles_xml, id_list):
            title_tag = article.find("articletitle")
            abstract_tag = article.find("abstract")
            date_tag = article.find("pubdate")
            author_tags = article.find_all("author")

            # Title
            title = title_tag.get_text(strip=True) if title_tag else "No title"

            # Abstract
            abstract = abstract_tag.get_text(separator=" ", strip=True) if abstract_tag else "No abstract available"

            # Authors
            authors = []
            for author in author_tags:
                last = author.find("lastname")
                fore = author.find("forename")
                if last and fore:
                    authors.append(f"{fore.get_text()} {last.get_text()}")
                elif last:
                    authors.append(last.get_text())
            authors = authors if authors else ["No authors listed"]

            # Publication Date
            pub_date = "No date"
            if date_tag:
                year = date_tag.find("year")
                month = date_tag.find("month")
                pub_date = f"{month.get_text()} {year.get_text()}" if year and month else year.get_text() if year else "No date"

            # PubMed Article URL
            url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"

            print(f"Article: {title}\n   - Authors: {authors}\n   - Date: {pub_date}\n   - URL: {url}\n")
            articles_info.append({
                "title": title,
                "abstract": abstract,
                "authors": authors,
                "publication_date": pub_date,
                "article_url": url
            })

        if not articles_info and use_mock_if_empty:
            print("No valid articles found, returning mock data.")
            return [{
                "title": "Simulated Study on Fever",
                "abstract": "This is a simulated abstract on the treatment of fever in adults.",
                "authors": ["John Doe", "Jane Smith"],
                "publication_date": "March 2024",
                "article_url": "https://pubmed.ncbi.nlm.nih.gov/12345678/"
            }]
        return articles_info

    except Exception as e:
        print(f"Error during PubMed fetch: {e}")
        if use_mock_if_empty:
            return [{
                "title": "Simulated Study on Fever",
                "abstract": "This is a simulated abstract on the treatment of fever in adults.",
                "authors": ["John Doe", "Jane Smith"],
                "publication_date": "March 2024",
                "article_url": "https://pubmed.ncbi.nlm.nih.gov/12345678/"
            }]
        else:
            return [{"message": f"Error: {e}"}]


In [6]:
fetch_pubmed_articles_with_metadata("fever and headache", max_results=3, use_mock_if_empty=True)

Found PubMed IDs: ['40293818', '40292140', '40292068']
Articles found in XML: 3
Article: Human intravenous immunoglobulins for recurrent pericarditis: a multicentre cohort study.
   - Authors: ['Valentino Collini', 'Francesco Venturelli', 'Razvan Berghi', 'Alessandro Andreis', 'Marzia De Biasio', 'Marco Merlo', 'Antonio Brucato', 'George Lazaros', 'Gianfranco Sinagra', 'Massimo Imazio']
   - Date: Apr 2025
   - URL: https://pubmed.ncbi.nlm.nih.gov/40293818/

Article: MRI findings in human rabies: A case report on the importance of neuroimaging when biological tests are inconclusive.
   - Authors: ['Zakaria Chahbi', 'Said Adnor', 'Soufiane Bigi', 'Mounir Salek', 'Soukaina Wakrim']
   - Date: Jul 2025
   - URL: https://pubmed.ncbi.nlm.nih.gov/40292140/

Article: COVID-19 Vaccination Adverse Events in Children: An Investigation with a Control Group in Tabriz Metropolitan City.
   - Authors: ['N Jafari', 'H Akbari', 'S Khayatzadeh', 'H Nobakht-Nojehdeh', 'P Sarbakhsh']
   - Date: Oct 2024


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_14896\2326587756.py:35: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(fetch_response.text, "html.parser")


[{'title': 'Human intravenous immunoglobulins for recurrent pericarditis: a multicentre cohort study.',
  'abstract': 'Based on limited data, intravenous immunoglobulins (IVIG) have been proposed as possible last therapeutic option for recurrent pericarditis (RP). The aim of this multicentre registry was to evaluate the efficacy and safety of IVIG in these patients after failure of other medical therapies. This multicentre cohort study enrolled consecutive patients with RP treated with IVIG. The primary outcome was the pericarditis recurrence rate after treatment with IVIG. A total of 43 patients (median age 41.7±14.4 years, 65.1% women) were included. The median duration of disease was 39 months (19-70) and the mean recurrences before IVIG was 5 (4-6). Most patients had elevated C-reactive protein (76.7%), pericardial effusion (72.1%) and fever (69.8%). IVIG were administered at a dose of 400-500 mg/kg/day for 5 consecutive days with repeated cycles, if needed. At discharge 40 (93%) p

In [3]:
!pip install lxml

   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.8 MB 5.7 MB/s eta 0:00:01
   -------- ------------------------------- 0.8/3.8 MB 4.8 MB/s eta 0:00:01
   ---------- ----------------------------- 1.0/3.8 MB 1.6 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/3.8 MB 1.6 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/3.8 MB 1.6 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/3.8 MB 1.6 MB/s eta 0:00:02
   ------------- -------------------------- 1.3/3.8 MB 754.4 kB/s eta 0:00:04
   ---------------- ----------------------- 1.6/3.8 MB 873.8 kB/s eta 0:00:03
   ------------------------ --------------- 2.4/3.8 MB 1.2 MB/s eta 0:00:02
   -------------------------------- ------- 3.1/3.8 MB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 1.6 MB/s eta 0:00:00
